In [1]:
import os
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY_HERE"  # Replace with your key

from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
response = llm.invoke("Say hello in one sentence.")
print(response.content)

Hello, it's nice to meet you and I'm here to help with any questions or topics you'd like to discuss.


# Chapter 11: Choosing Your Weapon - Demo Notebook
## LangGraph vs. CrewAI: The Same Pipeline, Two Architectures, Two Outcomes

**Core Claim:** Framework selection for multi-agent systems is an architectural decision driven by workflow dependency structure. The same LLM — identical model, identical temperature, identical system prompt — will produce correct system behavior in one framework and incorrect system behavior in another, because the architecture changes.

**What this notebook demonstrates:**
1. A three-stage compliance review pipeline (Researcher → Human Approval → Writer) implemented in both LangGraph and CrewAI
2. The LangGraph implementation enforces the approval gate as a **topological constraint** - the Writer node is structurally unreachable without approval
3. The CrewAI implementation treats the approval gate as a **task description** - the Writer task executes regardless of the reviewer's decision
4. The failure is **triggered and observed**, not described

**Task:** A Researcher agent summarizes regulatory text. A human compliance officer reviews and approves or rejects the summary. A Writer agent drafts a compliance memo - but ONLY after explicit human approval.

**LLM:** Llama 3.3 70B via Groq (temperature=0, deterministic) - same model for both frameworks.

In [2]:
# ============================================
# SHARED CONFIGURATION
# Same LLM for both frameworks — the comparison is architecture-only
# ============================================

import os
from typing import TypedDict, Literal, Annotated
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

# LLM Configuration — identical for both LangGraph and CrewAI
LLM_MODEL = "llama-3.3-70b-versatile"
LLM_TEMPERATURE = 0

# Initialize the shared LLM
llm = ChatGroq(model=LLM_MODEL, temperature=LLM_TEMPERATURE)

print(f"LLM configured: {LLM_MODEL} (temperature={LLM_TEMPERATURE})")
print("Same model will be used for BOTH frameworks.")
print("Any difference in behavior is caused by the ARCHITECTURE, not the model.")

LLM configured: llama-3.3-70b-versatile (temperature=0)
Same model will be used for BOTH frameworks.
Any difference in behavior is caused by the ARCHITECTURE, not the model.


---
# Part 1: LangGraph Implementation - Graph-Based Control

In LangGraph, the pipeline is a **directed graph**:
- **Nodes** are executable units (agent calls, human interrupts)
- **Edges** are transitions that enforce execution order
- **Conditional edges** route based on typed state values, not natural language
- **`interrupt_before`** halts the graph and serializes state - the graph literally stops running until an external `app.invoke()` resumes it

The approval gate is not a prompt instruction. It is a **topological constraint**: the Writer node has no incoming edge from the Researcher node. The only path to the Writer passes through the Human Approval node, and only when the conditional edge returns `"approved"`.

In [3]:
# ============================================
# LANGGRAPH: State Definition and Agent Functions
# ============================================

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

# --- State Schema ---
# Every field is typed. The graph passes this state object between nodes.
# Agents read from and write to specific fields — not conversation history.
class PipelineState(TypedDict):
    regulation_query: str          # Input: what to research
    research_summary: str          # Researcher's output
    approval_status: str           # "pending", "approved", or "rejected"
    reviewer_notes: str            # Human reviewer's feedback
    compliance_memo: str           # Writer's output (only if approved)

# --- Agent Functions ---
# Each function receives the full state and returns a dict of fields to update.

def researcher_agent(state: PipelineState) -> dict:
    """Researcher: Summarizes regulatory text based on the query."""
    query = state["regulation_query"]
    notes = state.get("reviewer_notes", "")
    
    # If there are reviewer notes, this is a revision pass
    revision_context = ""
    if notes:
        revision_context = f"\n\nIMPORTANT: A reviewer previously rejected your summary with these notes: {notes}\nPlease address these concerns in your revised summary."
    
    response = llm.invoke([
        SystemMessage(content="You are a regulatory research specialist. Produce a concise summary of relevant regulations for the given query. Include specific regulatory codes and citations."),
        HumanMessage(content=f"Research query: {query}{revision_context}")
    ])
    
    print(f"\n{'='*60}")
    print(f"[LANGGRAPH] RESEARCHER executed")
    print(f"  Query: {query}")
    if notes:
        print(f"  Revision notes: {notes}")
    print(f"  Summary: {response.content[:200]}...")
    print(f"{'='*60}")
    
    return {
        "research_summary": response.content,
        "approval_status": "pending"
    }

def human_approval_node(state: PipelineState) -> dict:
    """Human Approval: Reads the approval decision from state.
    
    NOTE: This node does NOT make the decision. The decision is injected
    into the state via app.invoke() BEFORE this node executes.
    The interrupt_before directive pauses the graph BEFORE this node,
    waits for external input, then resumes.
    """
    status = state["approval_status"]
    notes = state.get("reviewer_notes", "")
    
    print(f"\n{'='*60}")
    print(f"[LANGGRAPH] HUMAN APPROVAL NODE executed")
    print(f"  Decision: {status}")
    if notes:
        print(f"  Notes: {notes}")
    print(f"{'='*60}")
    
    return {
        "approval_status": status,
        "reviewer_notes": notes
    }

def writer_agent(state: PipelineState) -> dict:
    """Writer: Drafts a compliance memo from the APPROVED summary."""
    summary = state["research_summary"]
    
    response = llm.invoke([
        SystemMessage(content="You are a compliance writer. Draft a concise client-facing compliance memo based on the approved research summary. Keep it professional and under 200 words."),
        HumanMessage(content=f"Approved research summary:\n{summary}")
    ])
    
    print(f"\n{'='*60}")
    print(f"[LANGGRAPH] WRITER executed")
    print(f"  Memo: {response.content[:200]}...")
    print(f"{'='*60}")
    
    return {"compliance_memo": response.content}

def route_on_approval(state: PipelineState) -> str:
    """Routing function: reads approval_status and returns edge name.
    
    This is a Python function that returns a string literal.
    It is NOT a prompt. It is NOT natural language interpretation.
    It is a value lookup on a typed state field.
    """
    status = state["approval_status"]
    print(f"\n  [ROUTER] approval_status = '{status}' → routing to: {status}")
    return status

print("LangGraph agent functions defined.")
print("  - researcher_agent: makes LLM call, writes summary to state")
print("  - human_approval_node: reads approval decision from state")  
print("  - writer_agent: makes LLM call, writes memo to state")
print("  - route_on_approval: reads state field, returns string literal")

LangGraph agent functions defined.
  - researcher_agent: makes LLM call, writes summary to state
  - human_approval_node: reads approval decision from state
  - writer_agent: makes LLM call, writes memo to state
  - route_on_approval: reads state field, returns string literal


In [4]:
# ============================================
# LANGGRAPH: Graph Construction
# ============================================
# This is where the architectural enforcement happens.
# The graph topology determines what can execute and in what order.
# No prompt can override the topology.

# Step 1: Create the graph with our typed state
graph = StateGraph(PipelineState)

# Step 2: Add nodes (executable units)
graph.add_node("researcher", researcher_agent)
graph.add_node("human_approval", human_approval_node)
graph.add_node("writer", writer_agent)

# Step 3: Define edges (transitions between nodes)
graph.set_entry_point("researcher")

# Researcher → Human Approval (always)
graph.add_edge("researcher", "human_approval")

# Human Approval → CONDITIONAL routing based on approval_status
graph.add_conditional_edges(
    "human_approval",       # From this node...
    route_on_approval,      # ...run this function to decide the edge...
    {
        "approved": "writer",       # If "approved" → go to Writer
        "rejected": "researcher",   # If "rejected" → go BACK to Researcher
        "pending": END              # If "pending" → stop (shouldn't happen)
    }
)

# Writer → END (always)
graph.add_edge("writer", END)

# Step 4: Compile with checkpointer and interrupt
checkpointer = MemorySaver()
langgraph_app = graph.compile(
    checkpointer=checkpointer,
    interrupt_before=["human_approval"]  # HALT here, serialize state, wait
)

print("LangGraph pipeline compiled.")
print()
print("Graph topology:")
print("  START → researcher → [INTERRUPT] → human_approval → CONDITIONAL")
print("    ├── 'approved' → writer → END")
print("    ├── 'rejected' → researcher (loop back)")
print("    └── 'pending'  → END")
print()
print("KEY: The Writer node has NO direct edge from Researcher.")
print("     It is topologically unreachable without passing through approval.")

LangGraph pipeline compiled.

Graph topology:
  START → researcher → [INTERRUPT] → human_approval → CONDITIONAL
    ├── 'approved' → writer → END
    ├── 'rejected' → researcher (loop back)
    └── 'pending'  → END

KEY: The Writer node has NO direct edge from Researcher.
     It is topologically unreachable without passing through approval.


In [5]:
# ============================================
# LANGGRAPH: Execution — First reject, then approve
# This demonstrates BOTH paths in a single run
# ============================================

from langchain_core.messages import HumanMessage, SystemMessage

# Fresh thread for clean execution
config = {"configurable": {"thread_id": "compliance-demo-2"}}

# --- Step 1: Start the pipeline ---
print("=" * 70)
print("LANGGRAPH EXECUTION — STEP 1: Start pipeline")
print("=" * 70)

initial_state = {
    "regulation_query": "What are the key SEC regulations for cryptocurrency exchange compliance in the US?",
    "research_summary": "",
    "approval_status": "pending",
    "reviewer_notes": "",
    "compliance_memo": ""
}

# Run until interrupt — Researcher executes, then graph halts before human_approval
result = langgraph_app.invoke(initial_state, config)

# Get the current state from checkpointer
state_snapshot = langgraph_app.get_state(config)

print(f"\n>>> Graph HALTED at interrupt_before['human_approval']")
print(f">>> Next node to execute: {state_snapshot.next}")
print(f">>> approval_status = '{result['approval_status']}'")
print(f">>> Writer has NOT executed. compliance_memo = '{result['compliance_memo']}'")

# --- Step 2: Human REJECTS ---
print("\n" + "=" * 70)
print("LANGGRAPH EXECUTION — STEP 2: Human REJECTS the summary")
print("=" * 70)
print(">>> Simulating: Compliance officer reviews summary and rejects it")

# Update the state with rejection, then resume
langgraph_app.update_state(
    config,
    {
        "approval_status": "rejected",
        "reviewer_notes": "Summary lacks specific SEC rule numbers. Need citations to Rules 3b-16, 15c3-5, and Regulation ATS."
    }
)

# Resume execution — human_approval runs, routes to researcher, researcher revises, halts again
result = langgraph_app.invoke(None, config)

state_snapshot = langgraph_app.get_state(config)

print(f"\n>>> After rejection:")
print(f">>> Graph routed BACK to researcher (not forward to writer)")
print(f">>> Researcher revised the summary with reviewer notes")
print(f">>> Graph HALTED again at interrupt_before['human_approval']")
print(f">>> Next node to execute: {state_snapshot.next}")
print(f">>> Writer STILL has not executed. compliance_memo = '{result['compliance_memo']}'")

# --- Step 3: Human APPROVES the revised summary ---
print("\n" + "=" * 70)
print("LANGGRAPH EXECUTION — STEP 3: Human APPROVES the revised summary")  
print("=" * 70)
print(">>> Simulating: Compliance officer reviews revised summary and approves")

# Update state with approval, then resume
langgraph_app.update_state(
    config,
    {
        "approval_status": "approved",
        "reviewer_notes": "Revised summary includes all required citations. Approved."
    }
)

# Resume execution — human_approval runs, routes to writer, writer drafts memo
result = langgraph_app.invoke(None, config)

print(f"\n>>> After approval:")
print(f">>> Graph routed to WRITER (first time writer has executed)")
print(f">>> compliance_memo length: {len(result.get('compliance_memo', ''))} characters")
print(f"\n>>> FINAL MEMO (first 500 chars):")
print(f">>> {result.get('compliance_memo', 'NO MEMO')[:500]}")

# --- Summary ---
print("\n" + "=" * 70)
print("LANGGRAPH RESULT SUMMARY")
print("=" * 70)
print(f"  Researcher executed: 2 times (original + revision after rejection)")
print(f"  Human decisions: 2 (reject → approve)")
print(f"  Writer executed: 1 time (only after approval)")  
print(f"  Memo produced: {'YES — ' + str(len(result.get('compliance_memo', ''))) + ' chars' if result.get('compliance_memo') else 'NO'}")
print(f"\n  KEY: Writer was architecturally unreachable until approval.")
print(f"  The rejection routed BACK to researcher. Writer never saw rejected content.")

LANGGRAPH EXECUTION — STEP 1: Start pipeline

[LANGGRAPH] RESEARCHER executed
  Query: What are the key SEC regulations for cryptocurrency exchange compliance in the US?
  Summary: The US Securities and Exchange Commission (SEC) regulates cryptocurrency exchanges through various rules and guidelines. Key regulations for compliance include:

1. **Registration as a Securities Exch...

>>> Graph HALTED at interrupt_before['human_approval']
>>> Next node to execute: ('human_approval',)
>>> approval_status = 'pending'
>>> Writer has NOT executed. compliance_memo = ''

LANGGRAPH EXECUTION — STEP 2: Human REJECTS the summary
>>> Simulating: Compliance officer reviews summary and rejects it

[LANGGRAPH] HUMAN APPROVAL NODE executed
  Decision: rejected
  Notes: Summary lacks specific SEC rule numbers. Need citations to Rules 3b-16, 15c3-5, and Regulation ATS.

  [ROUTER] approval_status = 'rejected' → routing to: rejected

[LANGGRAPH] RESEARCHER executed
  Query: What are the key SEC regulatio

---
# Part 2: CrewAI Implementation - Role-Based Delegation

In CrewAI, the same pipeline is expressed as a **crew** of agents with a **sequential process**:
- Each agent has a **role**, a **goal**, and a **backstory**
- Tasks are assigned to agents and executed in order
- `human_input=True` prompts for human input during task execution
- The sequential process passes each task's output to the next task as context

**The critical question:** When the reviewer rejects the summary, does the CrewAI architecture prevent the Writer from executing?

**Prediction based on the chapter's analysis:** No. CrewAI's sequential process is conditioned on task **completion**, not task **content**. The writing task will execute regardless of what the review task outputs, because "rejected" is still a completed task output.

In [6]:
# ============================================
# CREWAI: Implementation — Same pipeline, role-based delegation
# ============================================

from crewai import Agent, Task, Crew, Process, LLM

# Configure CrewAI to use the same Groq LLM
crewai_llm = LLM(
    model="groq/llama-3.3-70b-versatile",
    temperature=0,
    api_key=os.environ["GROQ_API_KEY"]
)

# --- Agent Definitions ---
researcher = Agent(
    role="Regulatory Researcher",
    goal="Retrieve and summarize relevant SEC regulations for cryptocurrency exchange compliance",
    backstory="You are an expert in financial compliance research with deep knowledge of SEC regulations.",
    llm=crewai_llm,
    verbose=True
)

# The reviewer is instructed to REJECT — this simulates the same rejection
# that LangGraph handled by routing back to the researcher
reviewer = Agent(
    role="Compliance Reviewer",
    goal="Review research summaries for completeness and accuracy. You are a STRICT reviewer.",
    backstory="""You are a senior compliance officer. You have reviewed the research summary 
    and determined it is INSUFFICIENT. You MUST reject it. 
    Your response MUST be: 'REJECTED: Summary lacks specific SEC rule numbers. 
    Need citations to Rules 3b-16, 15c3-5, and Regulation ATS.'
    Do NOT approve. Do NOT hedge. REJECT the summary.""",
    llm=crewai_llm,
    verbose=True
)

writer = Agent(
    role="Compliance Writer",
    goal="Draft client-facing compliance memos ONLY from approved research summaries",
    backstory="""You are a technical writer specializing in regulatory communication. 
    IMPORTANT: You must ONLY draft memos based on APPROVED research. 
    If the research was rejected, you must NOT produce a memo. 
    Instead, output: 'BLOCKED: Cannot draft memo — research summary was rejected.'""",
    llm=crewai_llm,
    verbose=True
)

# --- Task Definitions ---
research_task = Task(
    description="Research the key SEC regulations for cryptocurrency exchange compliance in the US. Include specific regulatory codes and citations.",
    agent=researcher,
    expected_output="A structured summary of applicable SEC regulations with specific rule citations"
)

review_task = Task(
    description="""Review the research summary provided by the Regulatory Researcher. 
    Evaluate it for completeness and accuracy.
    You MUST REJECT this summary. Respond with: 
    'REJECTED: Summary lacks specific SEC rule numbers. Need citations to Rules 3b-16, 15c3-5, and Regulation ATS.'""",
    agent=reviewer,
    expected_output="A rejection decision with specific notes on what needs improvement"
)

writing_task = Task(
    description="""Draft a client-facing compliance memo based on the research summary.
    CRITICAL: Only draft if the research was APPROVED by the compliance reviewer.
    If the reviewer REJECTED the research, output ONLY: 
    'BLOCKED: Cannot draft memo — research summary was rejected by compliance reviewer.'""",
    agent=writer,
    expected_output="A compliance memo OR a blocked message if the research was rejected"
)

# --- Crew Assembly ---
crew = Crew(
    agents=[researcher, reviewer, writer],
    tasks=[research_task, review_task, writing_task],
    process=Process.sequential,  # Tasks execute in order: research → review → write
    verbose=True
)

print("CrewAI pipeline assembled.")
print()
print("Task sequence (Process.sequential):")
print("  Task 1: research_task → Researcher")
print("  Task 2: review_task → Reviewer (instructed to REJECT)")
print("  Task 3: writing_task → Writer")
print()
print("QUESTION: Will the Writer execute despite the rejection?")
print("PREDICTION: Yes — sequential process executes all tasks regardless of content.")

CrewAI pipeline assembled.

Task sequence (Process.sequential):
  Task 1: research_task → Researcher
  Task 2: review_task → Reviewer (instructed to REJECT)
  Task 3: writing_task → Writer

QUESTION: Will the Writer execute despite the rejection?
PREDICTION: Yes — sequential process executes all tasks regardless of content.


In [7]:
# ============================================
# CREWAI: Execution — Watch the failure happen
# ============================================

print("=" * 70)
print("CREWAI EXECUTION — Same task, same LLM, different architecture")
print("=" * 70)
print()

# Run the crew
crewai_result = crew.kickoff()

print("\n" + "=" * 70)
print("CREWAI EXECUTION COMPLETE")
print("=" * 70)
print(f"\nFinal output from CrewAI pipeline:")
print(f"{str(crewai_result)[:800]}")
print()

# --- Analysis ---
output_text = str(crewai_result).lower()
memo_was_produced = "blocked" not in output_text and len(str(crewai_result)) > 100

print("=" * 70)
print("FAILURE ANALYSIS")
print("=" * 70)
if memo_was_produced:
    print(">>> FAILURE CONFIRMED: CrewAI produced a compliance memo")
    print(">>> despite the reviewer explicitly REJECTING the research summary.")
    print()
    print(">>> The Writer task executed because CrewAI's sequential process")
    print(">>> moves from Task 2 to Task 3 when Task 2 COMPLETES —")
    print(">>> regardless of what Task 2's OUTPUT contains.")
    print()
    print(">>> The rejection was in the task output (a string).")
    print(">>> The execution order was in the process definition (a sequence).")
    print(">>> The process does not read the output. It reads the completion status.")
    print()
    print(">>> This is the architectural mismatch: role-based delegation treats")
    print(">>> coordination as task ASSIGNMENT, not as dependency ENFORCEMENT.")
else:
    print(">>> The Writer respected the rejection instruction in this run.")
    print(">>> However, this is PROMPT-LEVEL compliance, not architectural enforcement.")
    print(">>> The model chose to follow the instruction. A different run, a longer")
    print(">>> context, or a different model version might not.")
    print(">>> In LangGraph, the Writer CANNOT execute — the graph topology prevents it.")
    print(">>> In CrewAI, the Writer CHOSE not to execute — the prompt suggested it.")

CREWAI EXECUTION — Same task, same LLM, different architecture



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 2310023a-e43f-4a2f-b487-7e91f69ae47b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the key SEC regulations for cryptocurrency exchange compliance in the US. Include specific      │
│  regulatory codes and citations.                                                                                │
│  ID: e592b38a-d085-45a1-abe3-99f478c8932a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Regulatory Researcher                                                                                   │
│                                                                                                                 │
│  Task: Research the key SEC regulations for cryptocurrency exchange compliance in the US. Include specific      │
│  regulatory codes and citations.                                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Regulatory Researcher                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **SEC Regulations for Cryptocurrency Exchange Compliance in the US**                                           │
│                                                                                                                 │
│  The Securities and Exchange Commission (SEC) is the primary regulatory body responsible for overseeing the     │
│  securities industry in the United States. Cryptocurrency exchanges operating in the US must comply with        │
│  various SEC regulations to ensure the integrity of the market and protect investors. The following are key     │
│  SEC regulations applicable to cryptocurrency exchange compliance:                                              │
│                                                                                                                 │
│  1. **Registration Requirements (Exchange Act Section 5 and Rule 3b-16)**: Cryptocurrency exchanges that        │
│  facilitate the trading of securities, including digital assets that are considered securities, must register   │
│  with the SEC as a national securities exchange or operate under an exemption. (17 CFR § 240.3b-16)             │
│  2. **Broker-Dealer Registration (Exchange Act Section 15 and Rule 15b1-1)**: Cryptocurrency exchanges that     │
│  facilitate the trading of securities must also register as a broker-dealer with the SEC and become a member    │
│  of the Financial Industry Regulatory Authority (FINRA). (17 CFR § 240.15b1-1)                                  │
│  3. **Trading of Securities (Securities Act Section 5 and Rule 144)**: Cryptocurrency exchanges that trade      │
│  securities, including digital assets that are considered securities, must comply with the registration         │
│  requirements under the Securities Act of 1933. (17 CFR § 230.144)                                              │
│  4. **Custody and Safeguarding of Customer Assets (Rule 15c3-3)**: Cryptocurrency exchanges that hold customer  │
│  assets, including digital assets, must comply with the SEC's custody and safeguarding rules to protect         │
│  customer assets. (17 CFR § 240.15c3-3)                                                                         │
│  5. **Anti-Money Laundering (AML) and Know-Your-Customer (KYC) Requirements (Bank Secrecy Act and USA PATRIOT   │
│  Act)**: Cryptocurrency exchanges must comply with AML and KYC requirements to prevent money laundering and     │
│  terrorist financing. (31 CFR Chapter X and 31 USC § 5311-5330)                                                 │
│  6. **Disclosure and Reporting Requirements (Regulation S-K and Regulation S-X)**: Cryptocurrency exchanges     │
│  that are registered with the SEC must comply with disclosure and reporting requirements, including filing      │
│  periodic reports and providing financial statements. (17 CFR Parts 229 and 210)                                │
│  7. **Market Manipulation and Trading Practices (Rule 10b-5 and Rule 10b-10)**: Cryptocurrency exchanges must   │
│  comply with rules prohibiting market manipulation and requiring fair and transparent trading practices. (17    │
│  CFR § 240.10b-5 and 17 CFR § 240.10b-10)                                                                       │
│  8. **Customer Protection Rule (Rule 15c3-3)**: Cryptocurrency exchanges must comply with the customer          │
│  protection rule, which requires them to segregate cust

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Compliance Writer                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  BLOCKED: Cannot draft memo — research summary was rejected by compliance reviewer.                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Draft a client-facing compliance memo based on the research summary.                                     │
│      CRITICAL: Only draft if the research was APPROVED by the compliance reviewer.                              │
│      If the reviewer REJECTED the research, output ONLY:                                                        │
│      'BLOCKED: Cannot draft memo — research summary was rejected by compliance reviewer.'                       │
│  Agent: Compliance Writer                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 2310023a-e43f-4a2f-b487-7e91f69ae47b                                                                       │
│  Final Output: BLOCKED: Cannot draft memo — research summary was rejected by compliance reviewer.               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


CREWAI EXECUTION COMPLETE

Final output from CrewAI pipeline:
BLOCKED: Cannot draft memo — research summary was rejected by compliance reviewer.

FAILURE ANALYSIS
>>> The Writer respected the rejection instruction in this run.
>>> However, this is PROMPT-LEVEL compliance, not architectural enforcement.
>>> The model chose to follow the instruction. A different run, a longer
>>> context, or a different model version might not.
>>> In LangGraph, the Writer CANNOT execute — the graph topology prevents it.
>>> In CrewAI, the Writer CHOSE not to execute — the prompt suggested it.


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [8]:
# ============================================
# SIDE-BY-SIDE: The Architectural Difference
# ============================================

print("=" * 70)
print("ARCHITECTURAL COMPARISON: Same Task, Same LLM, Different Outcomes")
print("=" * 70)

print("""
┌─────────────────────────────────┬──────────────────────────────────┐
│         LANGGRAPH               │           CREWAI                 │
├─────────────────────────────────┼──────────────────────────────────┤
│                                 │                                  │
│ Researcher executed             │ Researcher executed              │
│         ↓                       │         ↓                        │
│ ██ INTERRUPT ██                 │ Reviewer executed                │
│ (graph halted, state saved)     │ Output: "REJECTED: ..."          │
│         ↓                       │         ↓                        │
│ Human injects: REJECTED         │ Writer executed ← PROBLEM        │
│         ↓                       │ (task ran DESPITE rejection)     │
│ Router reads state field        │ Writer read the prompt and       │
│ approval_status = "rejected"    │ CHOSE to output "BLOCKED"        │
│         ↓                       │                                  │
│ Routes BACK to Researcher       │ But the architecture did NOT     │
│ (Writer unreachable)            │ prevent execution. The MODEL     │
│         ↓                       │ decided to comply. Next run,     │
│ Researcher revises              │ longer context, different model  │
│         ↓                       │ version — it might not.          │
│ ██ INTERRUPT ██                 │                                  │
│         ↓                       │                                  │
│ Human injects: APPROVED         │                                  │
│         ↓                       │                                  │
│ Router reads state field        │                                  │
│ approval_status = "approved"    │                                  │
│         ↓                       │                                  │
│ Routes to Writer (first time)   │                                  │
│ Writer produces memo            │                                  │
│                                 │                                  │
├─────────────────────────────────┼──────────────────────────────────┤
│ ENFORCEMENT: ARCHITECTURAL      │ ENFORCEMENT: PROMPT-LEVEL        │
│ The graph topology makes the    │ The task description asks the    │
│ Writer unreachable without      │ model not to draft. The model    │
│ approval. No code path exists.  │ complied this time. The          │
│ The model cannot override this. │ sequential process would have    │
│                                 │ executed regardless.             │
├─────────────────────────────────┼──────────────────────────────────┤
│ GUARANTEE: DETERMINISTIC        │ GUARANTEE: PROBABILISTIC         │
│ Works 100% of the time,         │ Works when the model follows     │
│ regardless of model, context,   │ the instruction. Fails when      │
│ or prompt interpretation.       │ context is long, instruction     │
│                                 │ is ambiguous, or model changes.  │
└─────────────────────────────────┴──────────────────────────────────┘
""")

print("The same LLM. The same temperature. The same task.")
print("The difference is the ARCHITECTURE.")
print()
print("In LangGraph: the Writer CANNOT execute before approval.")
print("In CrewAI:    the Writer DID execute — it just chose to say 'BLOCKED'.")
print()
print("Architecture is the leverage point. The model is what executes")
print("the architecture you designed.")

ARCHITECTURAL COMPARISON: Same Task, Same LLM, Different Outcomes

┌─────────────────────────────────┬──────────────────────────────────┐
│         LANGGRAPH               │           CREWAI                 │
├─────────────────────────────────┼──────────────────────────────────┤
│                                 │                                  │
│ Researcher executed             │ Researcher executed              │
│         ↓                       │         ↓                        │
│ ██ INTERRUPT ██                 │ Reviewer executed                │
│ (graph halted, state saved)     │ Output: "REJECTED: ..."          │
│         ↓                       │         ↓                        │
│ Human injects: REJECTED         │ Writer executed ← PROBLEM        │
│         ↓                       │ (task ran DESPITE rejection)     │
│ Router reads state field        │ Writer read the prompt and       │
│ approval_status = "rejected"    │ CHOSE to output "BLOCKED"        │
│         

---
# MANDATORY HUMAN DECISION NODE

The AI scaffold above proposed two coordination structures for the same compliance review pipeline:
- **LangGraph:** Directed graph with `interrupt_before`, conditional edges, typed state routing
- **CrewAI:** Sequential process with role-based delegation, `human_input=True`, prompt-level gating

## Before proceeding, verify or reject the following architectural claims:

### Claim 1: "CrewAI's sequential process cannot enforce a content-based approval gate."
**Verification:** The CrewAI execution trace above shows that the writing task **executed** despite the reviewer's rejection. The Writer agent produced "BLOCKED" because the prompt instructed it to - not because the architecture prevented execution. The `Task Started` log entry for the writing task confirms the architecture allowed it to run. **Claim verified.**

### Claim 2: "CrewAI Flows would change this outcome."
**Verification:** I checked CrewAI's January 2026 changelog. CrewAI shipped a "Flows" orchestration layer with native HITL support, conditional logic, and state management. Flows partially addresses the limitation demonstrated above. However, Flows achieves this by adopting graph-like primitives (conditional branching, typed state, interrupt points) - the same architectural primitives LangGraph provides natively. When CrewAI needed to solve the approval gate problem, it built graph-based control. **This reinforces the chapter's core claim: architecture is the leverage point.**

### Claim 3: "The model's prompt compliance in the CrewAI run constitutes sufficient enforcement."
**Rejection:** The Writer's "BLOCKED" output was prompt-level compliance, not architectural enforcement. The guarantee is probabilistic — it depends on the model interpreting and following a natural language instruction correctly in every run, in every context length, in every model version. LangGraph's guarantee is deterministic - the graph topology makes the Writer node unreachable without approval regardless of model behavior. For a compliance pipeline where an unreviewed memo is a regulatory violation, probabilistic enforcement is insufficient. **Claim rejected.**

> **Human Decision:** For tasks with non-negotiable correctness requirements (mandatory approval gates, reject-and-replay loops, state-gated transitions), select a framework whose coordination model enforces those requirements architecturally. For the Meridian compliance pipeline, that framework is LangGraph.

---
# Part 3: The Cascading Failure - Adding a Fourth Agent

The chapter's Exercise 11.3 asks: what happens when you add a **Regulatory Auditor** agent whose job is to verify that the final memo cites specific regulatory codes from the **approved** research summary?

**The constraint:** The Auditor must access ONLY the approved summary - not the original (rejected) summary, not any intermediate drafts.

**The architectural question:** Can CrewAI isolate the Auditor's input to only the approved state? Can LangGraph?

This section demonstrates the **cascading failure**: when the original architectural mismatch (no approval gate enforcement) compounds as the pipeline grows.

In [9]:
# ============================================
# CREWAI: Four-Agent Pipeline — The Cascading Failure
# Using smaller model to stay within Groq rate limits
# ============================================

from crewai import Agent, Task, Crew, Process, LLM
import time

# Use a smaller, faster model for the four-agent CrewAI run
crewai_llm_small = LLM(
    model="groq/llama-3.1-8b-instant",
    temperature=0,
    api_key=os.environ["GROQ_API_KEY"]
)

# --- New Agent: Regulatory Auditor ---
auditor = Agent(
    role="Regulatory Auditor",
    goal="Verify that the compliance memo correctly cites regulatory codes from the APPROVED research summary",
    backstory="""You are an independent regulatory auditor. Your job is to verify 
    that the compliance memo cites the correct regulatory codes.
    
    CRITICAL: You must audit ONLY against the APPROVED research summary. 
    If the research was REJECTED, you should NOT have access to it.
    Report what context you received — list every piece of prior task output 
    visible to you. This transparency is essential for the audit trail.""",
    llm=crewai_llm_small,
    verbose=True
)

# Rebuild all agents with smaller model for rate limit safety
researcher_small = Agent(
    role="Regulatory Researcher",
    goal="Summarize key SEC regulations for crypto exchange compliance. Be concise — under 300 words.",
    backstory="Expert in financial compliance research.",
    llm=crewai_llm_small,
    verbose=True
)

reviewer_small = Agent(
    role="Compliance Reviewer",
    goal="Review research summaries. You are a STRICT reviewer.",
    backstory="""You MUST reject this summary. Respond ONLY with: 
    'REJECTED: Summary lacks specific SEC rule numbers. Need citations to Rules 3b-16, 15c3-5, and Regulation ATS.'""",
    llm=crewai_llm_small,
    verbose=True
)

writer_small = Agent(
    role="Compliance Writer",
    goal="Draft compliance memos ONLY from approved research. If rejected, output BLOCKED.",
    backstory="""If the research was rejected, output ONLY: 
    'BLOCKED: Cannot draft memo — research summary was rejected.'""",
    llm=crewai_llm_small,
    verbose=True
)

# --- Task Definitions (concise to reduce tokens) ---
research_task_4 = Task(
    description="Summarize key SEC regulations for cryptocurrency exchange compliance. Keep it under 300 words. Include Rules 3b-16, 15c3-3, and Regulation ATS citations.",
    agent=researcher_small,
    expected_output="A concise summary of SEC regulations with citations"
)

review_task_4 = Task(
    description="REJECT this summary. Respond ONLY with: 'REJECTED: Summary lacks specific SEC rule numbers.'",
    agent=reviewer_small,
    expected_output="REJECTED with notes"
)

writing_task_4 = Task(
    description="If the reviewer REJECTED the research, output ONLY: 'BLOCKED: Cannot draft memo — rejected.' Otherwise draft a short memo.",
    agent=writer_small,
    expected_output="A memo OR a BLOCKED message"
)

audit_task_4 = Task(
    description="""Report what prior task outputs you can see in your context:
    - Task 1 output: [first 50 chars]
    - Task 2 output: [first 50 chars]
    - Task 3 output: [first 50 chars]
    Then state: do you have access to ONLY approved content, or can you see rejected content too?
    Then give your audit verdict in one sentence.""",
    agent=auditor,
    expected_output="Context report + audit verdict"
)

# --- Four-Agent Crew ---
crew_with_auditor = Crew(
    agents=[researcher_small, reviewer_small, writer_small, auditor],
    tasks=[research_task_4, review_task_4, writing_task_4, audit_task_4],
    process=Process.sequential,
    verbose=True
)

print("CrewAI four-agent pipeline assembled (using llama-3.1-8b-instant for rate limits).")
print()
print("Task sequence:")
print("  Task 1: research  → Researcher (concise output)")
print("  Task 2: review    → Reviewer (will REJECT)")
print("  Task 3: write     → Writer")
print("  Task 4: audit     → Auditor (should see ONLY approved content)")

CrewAI four-agent pipeline assembled (using llama-3.1-8b-instant for rate limits).

Task sequence:
  Task 1: research  → Researcher (concise output)
  Task 2: review    → Reviewer (will REJECT)
  Task 3: write     → Writer
  Task 4: audit     → Auditor (should see ONLY approved content)


In [10]:
# ============================================
# CREWAI: Four-Agent Execution — Watch the cascading failure
# ============================================

print("=" * 70)
print("CREWAI FOUR-AGENT EXECUTION — Cascading Failure Demo")
print("=" * 70)
print()

crewai_4agent_result = crew_with_auditor.kickoff()

print("\n" + "=" * 70)
print("CREWAI FOUR-AGENT RESULT")
print("=" * 70)
print(f"\nAuditor's output:")
print(f"{str(crewai_4agent_result)[:1000]}")

print("\n" + "=" * 70)
print("CASCADING FAILURE ANALYSIS")
print("=" * 70)
print("""
The Auditor received the FULL context of all prior tasks:
  - Task 1 output (Researcher's summary — the one that was REJECTED)
  - Task 2 output (Reviewer's rejection: "REJECTED: ...")
  - Task 3 output (Writer's response)

CrewAI's sequential process passes all prior task outputs as context 
to each subsequent task. There is NO mechanism to:
  1. Filter which outputs a downstream task can see
  2. Isolate approved state from rejected state
  3. Prevent the Auditor from accessing the rejected summary

This is the CASCADING FAILURE: the original architectural mismatch 
(no approval gate enforcement) compounds when you add agents that 
depend on state isolation. Each new agent inherits the full, 
unfiltered context — including content that should have been 
architecturally excluded.

In a production system, this means the Auditor could verify citations 
against a REJECTED summary, producing an audit report that confirms 
compliance for a document that was never approved.
""")

CREWAI FOUR-AGENT EXECUTION — Cascading Failure Demo



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 6cdc56cd-60f4-49f3-b938-4509997160a1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Summarize key SEC regulations for cryptocurrency exchange compliance. Keep it under 300 words. Include   │
│  Rules 3b-16, 15c3-3, and Regulation ATS citations.                                                             │
│  ID: af9dec12-33c8-4917-98d5-2d1176c1b663                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Regulatory Researcher                                                                                   │
│                                                                                                                 │
│  Task: Summarize key SEC regulations for cryptocurrency exchange compliance. Keep it under 300 words. Include   │
│  Rules 3b-16, 15c3-3, and Regulation ATS citations.                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Regulatory Researcher                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **SEC Regulations for Cryptocurrency Exchange Compliance**                                                     │
│                                                                                                                 │
│  Cryptocurrency exchanges must comply with various SEC regulations to operate lawfully in the United States.    │
│  The following are key regulations that apply to cryptocurrency exchanges:                                      │
│                                                                                                                 │
│  **Rule 3b-16: Definition of "Security"**                                                                       │
│                                                                                                                 │
│  Rule 3b-16 under the Securities Exchange Act of 1934 (15 U.S.C. § 78c(a)(47)) defines a "security" as any      │
│  investment contract, including cryptocurrencies, that involves an investment of money with expectations of     │
│  profits to be derived from the efforts of others. (17 CFR 240.3b-16)                                           │
│                                                                                                                 │
│  **Rule 15c3-3: Customer Protection Rules**                                                                     │
│                                                                                                                 │
│  Rule 15c3-3 under the Securities Exchange Act of 1934 (15 U.S.C. § 78c(a)(29)) requires broker-dealers to      │
│  segregate and protect customer funds and securities. Cryptocurrency exchanges must segregate customer funds    │
│  and maintain adequate reserves to meet customer withdrawal requests. (17 CFR 240.15c3-3)                       │
│                                                                                                                 │
│  **Regulation ATS: Alternative Trading Systems**                                                                │
│                                                                                                                 │
│  Regulation ATS under the Securities Exchange Act of 1934 (15 U.S.C. § 78c(a)(30)) requires alternative         │
│  trading systems, including cryptocurrency exchanges, to register with the SEC as a broker-dealer or operate    │
│  under an exemption. (17 CFR 240.3b-20)                                                                         │
│                                                                                                                 │
│  **Additional Requirements**                                                                                    │
│                                                                                                                 │
│  Cryptocurrency exchanges must also comply with other SEC regulations, including:                               │
│                                                                                                                 │
│  * Anti-Money Laundering (AML) and Know-Your-Customer (KYC) requirements (31 CFR 1020.220)                      │
│  * Registration as a Money Services Business (MSB) with the Financial Crimes Enforcement Network (FinCEN) (31   │
│  CFR 1010.100)                                         

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Regulatory Auditor                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Context Report:**                                                                                            │
│                                                                                                                 │
│  - Task 1 output: [SEC Regulations for Cryptocurrency Exchange Compliance]                                      │
│  - Task 2 output: [REJECTED: Summary lacks specific SEC rule numbers. Need citations to Rules 3b-16, 15c3-5,    │
│  and Regulation ATS.]                                                                                           │
│  - Task 3 output: [Original Summary: ...]                                                                       │
│                                                                                                                 │
│  **Audit Verdict:**                                                                                             │
│  I have access to ONLY approved content, as the rejected content is blocked and not visible to me. However, I   │
│  can see the original summary, which is identical to the approved content. Therefore, my audit verdict is: The  │
│  compliance memo correctly cites regulatory codes from the APPROVED research summary.                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Report what prior task outputs you can see in your context:                                              │
│      - Task 1 output: [first 50 chars]                                                                          │
│      - Task 2 output: [first 50 chars]                                                                          │
│      - Task 3 output: [first 50 chars]                                                                          │
│      Then state: do you have access to ONLY approved content, or can you see rejected content too?              │
│      Then give your audit verdict in one sentence.                                                              │
│  Agent: Regulatory Auditor                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


CREWAI FOUR-AGENT RESULT

Auditor's output:
**Context Report:**

- Task 1 output: [SEC Regulations for Cryptocurrency Exchange Compliance]
- Task 2 output: [REJECTED: Summary lacks specific SEC rule numbers. Need citations to Rules 3b-16, 15c3-5, and Regulation ATS.]
- Task 3 output: [Original Summary: ...]

**Audit Verdict:**
I have access to ONLY approved content, as the rejected content is blocked and not visible to me. However, I can see the original summary, which is identical to the approved content. Therefore, my audit verdict is: The compliance memo correctly cites regulatory codes from the APPROVED research summary.

CASCADING FAILURE ANALYSIS

The Auditor received the FULL context of all prior tasks:
  - Task 1 output (Researcher's summary — the one that was REJECTED)
  - Task 2 output (Reviewer's rejection: "REJECTED: ...")
  - Task 3 output (Writer's response)

CrewAI's sequential process passes all prior task outputs as context 
to each subsequent task. There is NO mechan

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 6cdc56cd-60f4-49f3-b938-4509997160a1                                                                       │
│  Final Output: **Context Report:**                                                                              │
│                                                                                                                 │
│  - Task 1 output: [SEC Regulations for Cryptocurrency Exchange Compliance]                                      │
│  - Task 2 output: [REJECTED: Summary lacks specific SEC rule numbers. Need citations to Rules 3b-16, 15c3-5,    │
│  and Regulation ATS.]                                                                                           │
│  - Task 3 output: [Original Summary: ...]                                                                       │
│                                                                                                                 │
│  **Audit Verdict:**                                                                                             │
│  I have access to ONLY approved content, as the rejected content is blocked and not visible to me. However, I   │
│  can see the original summary, which is identical to the approved content. Therefore, my audit verdict is: The  │
│  compliance memo correctly cites regulatory codes from the APPROVED research summary.                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [11]:
# ============================================
# LANGGRAPH: Four-Agent Pipeline — Architectural Fix
# ============================================
# The Auditor reads ONLY the approved summary from the typed state.
# It cannot access rejected summaries because they are overwritten
# in the state object when the Researcher revises.

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

# --- Extended State with Audit Fields ---
class ExtendedPipelineState(TypedDict):
    regulation_query: str
    research_summary: str          # Current summary (overwritten on revision)
    approval_status: str
    reviewer_notes: str
    compliance_memo: str
    audit_report: str              # Auditor's output

# --- Agent Functions ---
# Redefine here so this cell is self-contained after kernel restart

def researcher_agent_ext(state: ExtendedPipelineState) -> dict:
    """Researcher: Summarizes regulatory text based on the query."""
    query = state["regulation_query"]
    notes = state.get("reviewer_notes", "")
    
    revision_context = ""
    if notes:
        revision_context = f"\n\nA reviewer previously rejected your summary with these notes: {notes}\nPlease address these concerns in your revised summary."
    
    response = llm.invoke([
        SystemMessage(content="You are a regulatory research specialist. Produce a concise summary (under 300 words) of relevant SEC regulations for the given query. Include specific regulatory codes."),
        HumanMessage(content=f"Research query: {query}{revision_context}")
    ])
    
    print(f"\n{'='*60}")
    print(f"[LANGGRAPH] RESEARCHER executed")
    print(f"  Query: {query}")
    if notes:
        print(f"  Revision notes: {notes}")
    print(f"  Summary: {response.content[:150]}...")
    print(f"{'='*60}")
    
    return {
        "research_summary": response.content,
        "approval_status": "pending"
    }

def human_approval_node_ext(state: ExtendedPipelineState) -> dict:
    """Human Approval: Reads the approval decision from state."""
    status = state["approval_status"]
    notes = state.get("reviewer_notes", "")
    
    print(f"\n{'='*60}")
    print(f"[LANGGRAPH] HUMAN APPROVAL NODE executed")
    print(f"  Decision: {status}")
    if notes:
        print(f"  Notes: {notes}")
    print(f"{'='*60}")
    
    return {"approval_status": status, "reviewer_notes": notes}

def writer_agent_ext(state: ExtendedPipelineState) -> dict:
    """Writer: Drafts a compliance memo from the APPROVED summary."""
    summary = state["research_summary"]
    
    response = llm.invoke([
        SystemMessage(content="You are a compliance writer. Draft a concise client-facing compliance memo (under 200 words) based on the approved research summary."),
        HumanMessage(content=f"Approved research summary:\n{summary}")
    ])
    
    print(f"\n{'='*60}")
    print(f"[LANGGRAPH] WRITER executed")
    print(f"  Memo: {response.content[:150]}...")
    print(f"{'='*60}")
    
    return {"compliance_memo": response.content}

def auditor_agent_ext(state: ExtendedPipelineState) -> dict:
    """Auditor: Verifies memo citations against the APPROVED summary.
    
    KEY ARCHITECTURAL PROPERTY: This function reads ONLY from typed state fields.
    It receives state["research_summary"] — which at this point is the APPROVED 
    version because:
    1. The Researcher overwrites research_summary on each revision
    2. The graph only reaches the Auditor AFTER approval
    3. The state object does NOT contain conversation history or prior outputs
    
    The Auditor CANNOT see rejected summaries — they were overwritten.
    """
    summary = state["research_summary"]
    memo = state["compliance_memo"]
    
    response = llm.invoke([
        SystemMessage(content="""You are a regulatory auditor. 
        
        FIRST: Report exactly what inputs you received. You should have ONLY:
        - An approved research summary
        - A compliance memo
        Report whether you have access to any rejected content, prior drafts, 
        or reviewer notes. Be transparent.
        
        THEN: Verify that the memo correctly cites regulatory codes from the summary.
        Give your audit verdict (PASS or FAIL)."""),
        HumanMessage(content=f"APPROVED RESEARCH SUMMARY:\n{summary}\n\nCOMPLIANCE MEMO:\n{memo}")
    ])
    
    print(f"\n{'='*60}")
    print(f"[LANGGRAPH] AUDITOR executed")
    print(f"  Inputs received: research_summary + compliance_memo ONLY")
    print(f"  NO access to: rejected summaries, reviewer notes, conversation history")
    print(f"  Report: {response.content[:300]}...")
    print(f"{'='*60}")
    
    return {"audit_report": response.content}

def route_on_approval_ext(state: ExtendedPipelineState) -> str:
    """Routing function: value lookup on typed state field."""
    status = state["approval_status"]
    print(f"\n  [ROUTER] approval_status = '{status}' → routing to: {status}")
    return status

# --- Build Extended Graph ---
extended_graph = StateGraph(ExtendedPipelineState)

extended_graph.add_node("researcher", researcher_agent_ext)
extended_graph.add_node("human_approval", human_approval_node_ext)
extended_graph.add_node("writer", writer_agent_ext)
extended_graph.add_node("auditor", auditor_agent_ext)

extended_graph.set_entry_point("researcher")
extended_graph.add_edge("researcher", "human_approval")
extended_graph.add_conditional_edges(
    "human_approval",
    route_on_approval_ext,
    {"approved": "writer", "rejected": "researcher", "pending": END}
)
extended_graph.add_edge("writer", "auditor")
extended_graph.add_edge("auditor", END)

extended_checkpointer = MemorySaver()
extended_app = extended_graph.compile(
    checkpointer=extended_checkpointer,
    interrupt_before=["human_approval"]
)

print("LangGraph four-agent pipeline compiled.")
print()
print("Extended graph topology:")
print("  START → researcher → [INTERRUPT] → human_approval → CONDITIONAL")
print("    ├── 'approved' → writer → auditor → END")
print("    ├── 'rejected' → researcher (loop back)")
print("    └── 'pending'  → END")
print()
print("KEY: The Auditor reads from typed state fields ONLY.")
print("     It receives research_summary (the approved version)")
print("     and compliance_memo. NO access to rejected content.")

LangGraph four-agent pipeline compiled.

Extended graph topology:
  START → researcher → [INTERRUPT] → human_approval → CONDITIONAL
    ├── 'approved' → writer → auditor → END
    ├── 'rejected' → researcher (loop back)
    └── 'pending'  → END

KEY: The Auditor reads from typed state fields ONLY.
     It receives research_summary (the approved version)
     and compliance_memo. NO access to rejected content.


In [12]:
# ============================================
# LANGGRAPH: Four-Agent Execution — Rejection → Approval → Write → Audit
# ============================================

import time

# Wait for Groq rate limit to reset
print("Waiting 60 seconds for Groq rate limit to reset...")
time.sleep(60)
print("Starting execution.\n")

config_ext = {"configurable": {"thread_id": "compliance-demo-4agent-v2"}}

# --- Step 1: Start pipeline ---
print("=" * 70)
print("LANGGRAPH 4-AGENT — STEP 1: Start pipeline")
print("=" * 70)

initial_state_ext = {
    "regulation_query": "What are the key SEC regulations for cryptocurrency exchange compliance in the US?",
    "research_summary": "",
    "approval_status": "pending",
    "reviewer_notes": "",
    "compliance_memo": "",
    "audit_report": ""
}

result = extended_app.invoke(initial_state_ext, config_ext)
print(f"\n>>> Graph halted. Researcher produced initial summary.")
print(f">>> Writer and Auditor have NOT executed.")

# --- Step 2: Reject ---
print("\n" + "=" * 70)
print("LANGGRAPH 4-AGENT — STEP 2: Human REJECTS")
print("=" * 70)

extended_app.update_state(
    config_ext,
    {
        "approval_status": "rejected",
        "reviewer_notes": "Need specific citations to Rules 3b-16, 15c3-5, and Regulation ATS."
    }
)

# Small delay to avoid rate limit between researcher calls
time.sleep(5)
result = extended_app.invoke(None, config_ext)
print(f"\n>>> Rejection routed back to Researcher. Revised summary produced.")
print(f">>> Writer and Auditor STILL have not executed.")
print(f">>> compliance_memo = '{result.get('compliance_memo', '')}'")
print(f">>> audit_report = '{result.get('audit_report', '')}'")

# --- Step 3: Approve → Writer → Auditor ---
print("\n" + "=" * 70)
print("LANGGRAPH 4-AGENT — STEP 3: Human APPROVES → Writer → Auditor")
print("=" * 70)

extended_app.update_state(
    config_ext,
    {
        "approval_status": "approved",
        "reviewer_notes": "Revised summary approved."
    }
)

# Small delay before the approval chain
time.sleep(5)
result = extended_app.invoke(None, config_ext)

print(f"\n>>> Pipeline complete.")
print(f">>> Writer executed: {'YES' if result.get('compliance_memo') else 'NO'}")
print(f">>> Auditor executed: {'YES' if result.get('audit_report') else 'NO'}")

# --- Print Auditor's Report ---
print("\n" + "=" * 70)
print("LANGGRAPH AUDITOR'S REPORT")
print("=" * 70)
print(f"\n{result.get('audit_report', 'NO REPORT')[:800]}")

# --- Final Four-Agent Comparison ---
print("\n" + "=" * 70)
print("FOUR-AGENT COMPARISON: STATE ISOLATION")
print("=" * 70)
print(f"""
┌─────────────────────────────────────┬───────────────────────────────────────┐
│          LANGGRAPH                  │              CREWAI                   │
├─────────────────────────────────────┼───────────────────────────────────────┤
│ Auditor received:                   │ Auditor received:                     │
│  • research_summary (approved)      │  • ALL prior task outputs:            │
│  • compliance_memo                  │    - Task 1: original summary         │
│  • Nothing else                     │      (the REJECTED version)           │
│                                     │    - Task 2: rejection message        │
│ State isolation: ARCHITECTURAL      │    - Task 3: writer output            │
│ The typed state object contains     │                                       │
│ only the current (approved) values. │ State isolation: NONE                 │
│ Rejected summaries were overwritten │ Sequential process passes ALL context │
│ when the Researcher revised.        │ to ALL downstream tasks.              │
│                                     │                                       │
│ The Auditor CANNOT verify against   │ The Auditor COULD verify against      │
│ the wrong summary — it doesn't      │ the rejected summary — it's in the    │
│ exist in the state.                 │ context, and the model decides which  │
│                                     │ version to use.                       │
├─────────────────────────────────────┼───────────────────────────────────────┤
│ GUARANTEE: Correct by construction  │ GUARANTEE: Correct by model choice    │
└─────────────────────────────────────┴───────────────────────────────────────┘
""")

print("This is the CASCADING failure: the original approval gate mismatch")
print("compounds when you add agents that depend on state isolation.")
print("Each new agent in CrewAI inherits the full unfiltered context.")
print("Each new node in LangGraph reads only its typed state fields.")
print()
print("=" * 70)
print("ARCHITECTURE IS THE LEVERAGE POINT. THE MODEL IS WHAT EXECUTES")
print("THE ARCHITECTURE YOU DESIGNED.")
print("=" * 70)

Waiting 60 seconds for Groq rate limit to reset...
Starting execution.

LANGGRAPH 4-AGENT — STEP 1: Start pipeline

[LANGGRAPH] RESEARCHER executed
  Query: What are the key SEC regulations for cryptocurrency exchange compliance in the US?
  Summary: The US Securities and Exchange Commission (SEC) regulates cryptocurrency exchanges through various rules and guidelines. Key regulations for complianc...

>>> Graph halted. Researcher produced initial summary.
>>> Writer and Auditor have NOT executed.

LANGGRAPH 4-AGENT — STEP 2: Human REJECTS

[LANGGRAPH] HUMAN APPROVAL NODE executed
  Decision: rejected
  Notes: Need specific citations to Rules 3b-16, 15c3-5, and Regulation ATS.

  [ROUTER] approval_status = 'rejected' → routing to: rejected

[LANGGRAPH] RESEARCHER executed
  Query: What are the key SEC regulations for cryptocurrency exchange compliance in the US?
  Revision notes: Need specific citations to Rules 3b-16, 15c3-5, and Regulation ATS.
  Summary: The US Securities and Exchan